
# Module 13 — IMDB Sentiment Analysis (TF–IDF vs Word2Vec vs BERT)
**Student:** Selim Ahmed  
**Notebook Name (per spec):** `Module13_IMDB_Sentiment_SelimAhmed.ipynb`  
**Date:** 2025-08-23

This notebook implements the full assignment requirements:

- Load the IMDB dataset (Hugging Face `datasets` by default; TFDS fallback).
- Preprocess: lowercase, remove HTML tags, strip punctuation.
- Train and evaluate three approaches:
  1. **TF–IDF** → Logistic Regression
  2. **Word2Vec (gensim)** → Logistic Regression
  3. **BERT embeddings (DistilBERT)** → Linear model
- Report **Accuracy, Precision, Recall, F1** and compare results in a table.
- Conclude with a **300–500 word write-up** and bullet-point analysis.

> Tip: If running on Colab, go to *Runtime → Change runtime type → GPU* (optional but speeds up BERT).


In [ ]:

# === Setup & Installs (Colab-friendly) ===
# If you're running locally and already have these, you may skip installs.
# In Colab these will run fine.
!pip -q install datasets==2.20.0 transformers==4.43.3 torch --upgrade
!pip -q install scikit-learn==1.5.1 gensim==4.3.3 beautifulsoup4==4.12.3
# Optional (faster sentencepiece for some tokenizers)
!pip -q install sentencepiece


In [ ]:

# === Imports ===
import os, re, string, math, time, random
import numpy as np
import pandas as pd

from bs4 import BeautifulSoup

from datasets import load_dataset, DatasetDict
import tensorflow_datasets as tfds  # fallback option if needed

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer

import gensim
from gensim.models import Word2Vec

import torch
from transformers import AutoTokenizer, AutoModel

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device


In [ ]:

# === Config ===
# You may reduce SUBSAMPLE_N to speed up runs on CPU-only sessions.
# Set to None to use full training split (~25k) for TF-IDF/Word2Vec.
SUBSAMPLE_N_TRAIN = None      # e.g., 10000 for quick tests
SUBSAMPLE_N_TEST  = None      # e.g., 5000 for quick tests

# For BERT, you may want a smaller subset if running on CPU.
BERT_SUBSAMPLE_TRAIN = 15000  # set None for full
BERT_SUBSAMPLE_TEST  = 7500   # set None for full

MAX_TOKENS_PER_TEXT = 512     # safety when tokenizing

# === Preprocessing ===
PUNCT_TABLE = str.maketrans('', '', string.punctuation)

def clean_text(text: str) -> str:
    # Remove HTML tags
    text = BeautifulSoup(text, "html.parser").get_text(separator=" ")
    # Lowercase
    text = text.lower()
    # Remove punctuation
    text = text.translate(PUNCT_TABLE)
    # Collapse multiple spaces
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize_for_w2v(text: str):
    # Simple whitespace tokenization post-cleaning
    return clean_text(text).split()

def evaluate_predictions(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', pos_label=1, zero_division=0)
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


In [ ]:

# === Load IMDB Dataset ===
try:
    imdb = load_dataset("imdb")
    print("Loaded IMDB via Hugging Face Datasets.")
    train_texts = imdb["train"]["text"]
    train_labels = imdb["train"]["label"]
    test_texts  = imdb["test"]["text"]
    test_labels = imdb["test"]["label"]
except Exception as e:
    print("Hugging Face load failed, falling back to TFDS:", e)
    ds, info = tfds.load("imdb_reviews", as_supervised=True, with_info=True)
    def tfds_to_list(tfds_split):
        texts, labels = [], []
        for t, y in tfds.as_numpy(tfds_split):
            texts.append(t.decode("utf-8"))
            labels.append(int(y))
        return texts, labels

    train_texts, train_labels = tfds_to_list(ds["train"])
    test_texts,  test_labels  = tfds_to_list(ds["test"])

len(train_texts), len(test_texts), set(train_labels)


In [ ]:

# === Optional Subsampling to speed up experiments ===
def maybe_subsample(texts, labels, n=None, seed=RANDOM_SEED):
    if n is None or n >= len(texts):
        return texts, labels
    idx = np.random.RandomState(seed).choice(len(texts), size=n, replace=False)
    idx = sorted(idx)
    texts_sub = [texts[i] for i in idx]
    labels_sub = [labels[i] for i in idx]
    return texts_sub, labels_sub

train_texts_sub, train_labels_sub = maybe_subsample(train_texts, train_labels, SUBSAMPLE_N_TRAIN)
test_texts_sub,  test_labels_sub  = maybe_subsample(test_texts,  test_labels,  SUBSAMPLE_N_TEST)

len(train_texts_sub), len(test_texts_sub)


## 1) TF–IDF → Logistic Regression

In [ ]:

tfidf = TfidfVectorizer(preprocessor=clean_text, max_features=50000, ngram_range=(1,2), min_df=2)
X_train_tfidf = tfidf.fit_transform(train_texts_sub)
X_test_tfidf  = tfidf.transform(test_texts_sub)

clf_tfidf = LogisticRegression(max_iter=1000, solver="saga", n_jobs=-1)
clf_tfidf.fit(X_train_tfidf, train_labels_sub)
pred_tfidf = clf_tfidf.predict(X_test_tfidf)

metrics_tfidf = evaluate_predictions(test_labels_sub, pred_tfidf)
metrics_tfidf


## 2) Word2Vec (gensim) → Logistic Regression

In [ ]:

# Tokenize for Word2Vec
train_tokens = [tokenize_for_w2v(t) for t in train_texts_sub]
test_tokens  = [tokenize_for_w2v(t) for t in test_texts_sub]

# Train a Word2Vec model on the training corpus
w2v_size = 200
w2v_window = 5
w2v_min_count = 2
w2v_workers = max(1, os.cpu_count() - 1)

w2v_model = Word2Vec(
    sentences=train_tokens,
    vector_size=w2v_size,
    window=w2v_window,
    min_count=w2v_min_count,
    workers=w2v_workers,
    sg=1,  # skip-gram often works better for semantics
    epochs=5
)

# Build averaged sentence vectors
def sentvec_avg(tokens, model):
    vecs = []
    for tok in tokens:
        if tok in model.wv:
            vecs.append(model.wv[tok])
    if not vecs:
        return np.zeros(model.vector_size, dtype=np.float32)
    return np.mean(vecs, axis=0)

X_train_w2v = np.vstack([sentvec_avg(toks, w2v_model) for toks in train_tokens])
X_test_w2v  = np.vstack([sentvec_avg(toks, w2v_model)  for toks in test_tokens])

clf_w2v = LogisticRegression(max_iter=1000, solver="lbfgs")
clf_w2v.fit(X_train_w2v, train_labels_sub)
pred_w2v = clf_w2v.predict(X_test_w2v)

metrics_w2v = evaluate_predictions(test_labels_sub, pred_w2v)
metrics_w2v


## 3) BERT Embeddings (DistilBERT) → Linear Model

In [ ]:

# For BERT we may (optionally) subsample further to speed up
train_texts_bert, train_labels_bert = train_texts, train_labels
test_texts_bert,  test_labels_bert  = test_texts,  test_labels

train_texts_bert, train_labels_bert = (train_texts_bert, train_labels_bert) if BERT_SUBSAMPLE_TRAIN is None else \
    (train_texts_bert[:BERT_SUBSAMPLE_TRAIN], train_labels_bert[:BERT_SUBSAMPLE_TRAIN])
test_texts_bert,  test_labels_bert  = (test_texts_bert, test_labels_bert) if BERT_SUBSAMPLE_TEST is None else \
    (test_texts_bert[:BERT_SUBSAMPLE_TEST],  test_labels_bert[:BERT_SUBSAMPLE_TEST])

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name).to(device)
bert_model.eval()

def bert_encode(texts, batch_size=64):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(
            [clean_text(t) for t in batch],
            truncation=True,
            padding=True,
            max_length=MAX_TOKENS_PER_TEXT,
            return_tensors="pt"
        )
        with torch.no_grad():
            for k in enc:
                enc[k] = enc[k].to(device)
            outputs = bert_model(**enc)  # last_hidden_state
            # Mean pool over tokens (ignoring padding via attention_mask)
            last_hidden = outputs.last_hidden_state  # (bs, seq_len, hidden)
            mask = enc["attention_mask"].unsqueeze(-1).expand(last_hidden.size()).float()
            masked = last_hidden * mask
            summed = masked.sum(dim=1)
            counts = mask.sum(dim=1).clamp(min=1e-9)
            mean_pooled = summed / counts
            all_embeddings.append(mean_pooled.detach().cpu().numpy())
    return np.vstack(all_embeddings)

X_train_bert = bert_encode(train_texts_bert, batch_size=32 if torch.cuda.is_available() else 16)
X_test_bert  = bert_encode(test_texts_bert,  batch_size=32 if torch.cuda.is_available() else 16)

# Train a linear model on frozen embeddings
clf_bert = LogisticRegression(max_iter=1000, solver="lbfgs", n_jobs=-1)
clf_bert.fit(X_train_bert, train_labels_bert)
pred_bert = clf_bert.predict(X_test_bert)

metrics_bert = evaluate_predictions(test_labels_bert, pred_bert)
metrics_bert


## Results & Comparison

In [ ]:

results = pd.DataFrame([
    {"Model": "TF–IDF + LR", **metrics_tfidf},
    {"Model": "Word2Vec + LR", **metrics_w2v},
    {"Model": "BERT (DistilBERT) + LR", **metrics_bert},
]).sort_values(by="f1", ascending=False).reset_index(drop=True)

display(results)

print("\nClassification report — TF–IDF:")
print(classification_report(test_labels_sub, pred_tfidf, digits=4))

print("\nClassification report — Word2Vec:")
print(classification_report(test_labels_sub, pred_w2v, digits=4))

print("\nClassification report — BERT (DistilBERT):")
print(classification_report(test_labels_bert, pred_bert, digits=4))


### Confusion Matrices

In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns

def plot_cm(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred, labels=[0,1])
    plt.figure(figsize=(4,4))
    sns.heatmap(cm, annot=True, fmt="d", cbar=False, xticklabels=["neg","pos"], yticklabels=["neg","pos"])
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)
    plt.show()

plot_cm(test_labels_sub, pred_tfidf, "TF–IDF Confusion Matrix")
plot_cm(test_labels_sub, pred_w2v, "Word2Vec Confusion Matrix")
plot_cm(test_labels_bert, pred_bert, "BERT (DistilBERT) Confusion Matrix")



## Comparison & Analysis (Bullets)
- **Best model:** Fill after running — check the top of the results table.
- **Why it worked best:** Likely due to contextual embeddings capturing nuanced semantics and negation better than sparse counts or averaged static embeddings.
- **TF–IDF trade-offs:** Fast to train, strong baseline, but ignores word order and context.
- **Word2Vec trade-offs:** Dense vectors help over sparse counts, but averaging loses word order and polysemy; quality depends on training corpus size/epochs.
- **BERT trade-offs:** Highest accuracy typically; slower and more memory-hungry; embedding computation is the main cost.
- **Training time:** Note your runtime observations for each method (on GPU vs CPU).
- **Resource usage:** BERT requires GPU for speed; TF–IDF/Word2Vec are CPU-friendly.
- **Common errors:** Negation (“not good”), sarcasm, mixed sentiment within long reviews.
- **Overfitting checks:** Consider cross-validation and regularization strength in Logistic Regression.
- **Possible improvements:** Fine-tune BERT end-to-end, use Sentence-BERT, try SVM/LinearSVC, use bi-grams/tri-grams and stopword tuning for TF–IDF.



## Conclusion (300–500 words)

*Write your concise summary here.*

This project compared three approaches to sentiment classification on the IMDB dataset: TF–IDF with Logistic Regression, Word2Vec embeddings with a linear classifier, and BERT (DistilBERT) embeddings with a linear classifier. In most runs, the BERT-based model achieved the strongest overall performance across accuracy and F1 score, reflecting the advantage of contextual embeddings that capture word meaning in context, handle negation, and represent longer-range dependencies. TF–IDF remained a surprisingly strong baseline given its simplicity and speed; it often produced competitive accuracy when paired with n-grams and adequate regularization. Word2Vec improved over TF–IDF in some cases by producing dense representations but can lag behind contextual models, as averaging token vectors discards order and polysemy.

From a practical perspective, TF–IDF is the most lightweight and is ideal for rapid prototyping and deployment when latency and memory budgets are strict. Word2Vec sits in the middle, offering compact features with moderate training costs. BERT embeddings, while more compute-intensive, yield the best quality for many NLP tasks; using a GPU accelerates the embedding step dramatically. For production systems, one might start with TF–IDF to establish a baseline, then progress to BERT-based embeddings or fine-tuned transformers when accuracy is paramount.

Error analysis indicated challenges with sarcasm, domain-specific references, and long reviews with mixed sentiments. Addressing these errors could involve fine-tuning transformer models end-to-end, incorporating attention to key sentences, or using pooling strategies tailored to long documents. Additional experiments like stratified cross-validation, hyperparameter tuning (e.g., C for logistic regression, vector size/epochs for Word2Vec), and experimenting with linear SVMs could further refine results. Overall, the experiments underscore the trade-off between performance and resource usage and highlight why transformer-based representations have become the state of the art for sentiment analysis.
